In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub
import os

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.optimizers import Adam


e:\CS 180 CNN\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load new dataset
base_path = r"E:\CS 180 CNN\AI vs Human 2025 Data"
train_csv_path = os.path.join(base_path, "train.csv")
test_csv_path = os.path.join(base_path, "test.csv")

# train_img_dir = os.path.join(base_path, "train_data")
# test_img_dir = os.path.join(base_path, "test_data_v2")

train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

test_df = test_df.rename(columns={'id': 'file_name'})

train_df['file_name'] = base_path + "\\" + train_df['file_name'].astype(str) 
test_df['file_name'] = base_path + "\\" + test_df['file_name'].astype(str)

# old dataset
old_train_real_dir = r"C:\Users\Robin\.cache\kagglehub\datasets\antorbosuantu\asa-real-fake-dataset\versions\4\Final Dataset\Train\Train_Real"
old_train_fake_dir = r"C:\Users\Robin\.cache\kagglehub\datasets\antorbosuantu\asa-real-fake-dataset\versions\4\Final Dataset\Train\Train_Fake"
old_test_real_dir = r"C:\Users\Robin\.cache\kagglehub\datasets\antorbosuantu\asa-real-fake-dataset\versions\4\Final Dataset\Test\Test_Real"
old_test_fake_dir = r"C:\Users\Robin\.cache\kagglehub\datasets\antorbosuantu\asa-real-fake-dataset\versions\4\Final Dataset\Test\Test_Fake"

old_train_real_paths = [os.path.join(old_train_real_dir, f) for f in os.listdir(old_train_real_dir)]
old_train_fake_paths = [os.path.join(old_train_fake_dir, f) for f in os.listdir(old_train_fake_dir)]
old_test_real_paths = [os.path.join(old_test_real_dir, f) for f in os.listdir(old_test_real_dir)]
old_test_fake_paths = [os.path.join(old_test_fake_dir, f) for f in os.listdir(old_test_fake_dir)]

# 0 = Real, 1 = Fake
df_old_train_real = pd.DataFrame({'file_name': old_train_real_paths, 'label': 0})
df_old_train_fake = pd.DataFrame({'file_name': old_train_fake_paths, 'label': 1})
df_old_test_real = pd.DataFrame({'file_name': old_test_real_paths, 'label': 0})
df_old_test_fake = pd.DataFrame({'file_name': old_test_fake_paths, 'label': 1})

master_df = pd.concat([
    train_df, 
    df_old_train_real, df_old_train_fake,
    df_old_test_real, df_old_test_fake
], ignore_index=True)

master_df = master_df.dropna(subset=['label'])

train_df_final, test_df_final = train_test_split(
    master_df, test_size=0.2, random_state=42, stratify=master_df['label']
)

print(f"Total Master Images: {len(master_df)}")
print(f"Training Images (80%): {len(train_df_final)}")
print(f"Validation Images (20%): {len(test_df_final)}\n")


class_counts = train_df_final['label'].value_counts()
total_images = len(train_df_final)

weight_for_0 = (1 / class_counts[0]) * (total_images / 2.0)
weight_for_1 = (1 / class_counts[1]) * (total_images / 2.0)

class_weights = {0: weight_for_0, 1: weight_for_1}
print(f"Master Class Weights: {class_weights}\n")


train_paths = train_df_final['file_name'].values
train_labels = train_df_final['label'].values

test_paths = test_df_final['file_name'].values
test_labels = test_df_final['label'].values

Total Master Images: 89950
Training Images (80%): 71960
Validation Images (20%): 17990

Master Class Weights: {0: 1.0, 1: 1.0}



In [3]:
def load_and_resize(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [224, 224])
    return img, label

batch_size = 16
AUTOTUNE = tf.data.AUTOTUNE

# build training dataset
train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = train_ds.map(load_and_resize, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.batch(batch_size)

test_ds = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
test_ds = test_ds.map(load_and_resize, num_parallel_calls=AUTOTUNE)
test_ds = test_ds.batch(batch_size)

# augment data
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=AUTOTUNE
).prefetch(buffer_size=AUTOTUNE)

test_ds = test_ds.map(
    lambda x, y: (preprocess_input(x), y)
).prefetch(AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [4]:
#setup ResNET-50
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = models.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu"),
    layers.Dropout(0.4),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(1, activation="sigmoid")
])

In [5]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
# run model
optimizer = Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999)

model.compile(
    optimizer=optimizer,
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()
epochs = 10

early_stop = EarlyStopping(patience=3, restore_best_weights=True)

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=epochs,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential (Sequential)     (None, 224, 224, 3)       0         
                                                                 
 resnet50 (Functional)       (None, 7, 7, 2048)        23587712  
                                                                 
 global_average_pooling2d (G  (None, 2048)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 256)               524544    
                                                                 
 dropout (Dropout)           (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 128)               32896     
                                                      

In [6]:
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.3, patience=2, min_lr=1e-6)
]

history_finetune = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10,
    callbacks=callbacks,
    class_weight=class_weights
)

Epoch 1/10
4498/4498 [==============================] - 920s 203ms/step - loss: 0.1692 - accuracy: 0.9347 - val_loss: 0.1980 - val_accuracy: 0.9291 - lr: 1.0000e-05
Epoch 2/10
4498/4498 [==============================] - 912s 203ms/step - loss: 0.1029 - accuracy: 0.9616 - val_loss: 0.2273 - val_accuracy: 0.9252 - lr: 1.0000e-05
Epoch 3/10
4498/4498 [==============================] - 911s 203ms/step - loss: 0.0746 - accuracy: 0.9712 - val_loss: 0.2207 - val_accuracy: 0.9344 - lr: 1.0000e-05
Epoch 4/10
4498/4498 [==============================] - 915s 204ms/step - loss: 0.0549 - accuracy: 0.9793 - val_loss: 0.2687 - val_accuracy: 0.9239 - lr: 3.0000e-06


In [7]:
y_true = np.concatenate([y for x, y in test_ds])
y_pred = model.predict(test_ds)
y_pred = (y_pred > 0.5).astype(int)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

model.save("Veritas_Model_May03_2.h5")

1125/1125 [==============================] - 86s 76ms/step
[[8077  918]
 [ 358 8637]]
              precision    recall  f1-score   support

           0       0.96      0.90      0.93      8995
           1       0.90      0.96      0.93      8995

    accuracy                           0.93     17990
   macro avg       0.93      0.93      0.93     17990
weighted avg       0.93      0.93      0.93     17990

